**Legacy notebook.** Self-contained analysis code that predates the `src/mrvf` library and has not been ported to it. Kept for provenance and because it still produces figures in `results/`. Paths were updated to the `results/` layout; the next cell sets the working directory to the repository root, so run it from anywhere.

For the maintained pipeline see `notebooks/01_train_triple_regime.ipynb` and `notebooks/02_evaluate_rmse_vs_snr.ipynb`.

In [ ]:
import os
from pathlib import Path
# run from the repository root so ./results/... and ../subsamples resolve
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "mrvf").is_dir())
os.chdir(_root)

# Dual-Regime T2 Features for MRvF Deep Learning

## Motivation and Physics

### Why the combined Ni Eq. 3 T2 estimate underestimates true T2

In the GESFIDE sequence, the signal in each regime follows:
```
Part A (FID):       S_A(t) = S0 · exp(−R2*_A · t)    R2*_A = R2 + R2'   [always decays]
Part B (rephasing): S_B(t) = S0_B · exp(−R2*_B · t)  R2*_B = R2 − R2'   [can refocus]
```

Ni Eq. 3 combines them: `R2 = (R2*_A + R2*_B)/2` — R2' cancels algebraically.

**The failure mode**: when R2' ≥ R2 (long T2, high deoxygenation), Part B **rephases**
(signal amplitude increases toward the spin echo). Log-linear OLS cannot fit this arch;
it estimates R2*_B ≈ 0 instead of the true negative value, causing R2 overestimation
and systematic T2 **underestimation** (negative bias).

### The new approach: two separate regime features

Instead of combining into one biased T2 estimate, we pass **both regime slopes separately**
as additional network inputs:

| Feature | Formula | Range | Bias direction |
|---------|---------|-------|---------------|
| R2*_A   | `−slope_A`  ≥ 0 | ~[5, 50] s⁻¹ | encodes R2 + R2' → T2_A < true T2 |
| R2*_B   | `−slope_B`, can be < 0 | ~[−30, 20] s⁻¹ | encodes R2 − R2' → T2_B > true T2 |

**Key insight**: together R2*_A and R2*_B independently span R2 and R2':
```
R2  = (R2*_A + R2*_B) / 2     [true T2 = 1/R2]
R2' = (R2*_A − R2*_B) / 2     [encodes SO2/CBV sensitivity]
```

The network receives the 40-echo signal **plus** both regime statistics, giving it
explicit access to both the reversible and irreversible relaxation components —
richer than either a single biased T2 or a single R2_eff scalar.

**Network input**: 40 L2-normalised GESFIDE echoes + scaled R2*_A + scaled R2*_B = **42 dims**  
**Network output**: SO₂, CBV, R, T2 (4 parameters)

---

## 1. Imports & GPU

In [ ]:
import os, json, time
import numpy as np
import scipy.io as sio
import h5py
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    device = torch.device('cpu')
    print('CPU only')
print(f'PyTorch {torch.__version__}')

## 2. Configuration

In [ ]:
CONFIG = {
    # --- Data paths (adjust to your directory layout) ---
    'dict_base_path':     '../subsamples/subsamples_v3',
    'param_path':         '../subsamples/subsamples_v3/QuasiRand_par_t2_200.mat',
    'noisefree_sig_path': '../subsamples/subsamples_v3/QuasiRand_t2_200.mat',
    'echotimes_path':     '../echotimes.mat',
    'output_dir':         './results/dual_regime_results_v1',
    'dict_key':           'Dico40_save',
    'param_key':          'par_save',

    # --- Parameter space (same as Option 3 / ISBI paper) ---
    'param_mins':   np.array([0.0,    0.0025,  1.0e-6,  0.050]),   # SO2, CBV, R(m), T2(s)
    'param_maxs':   np.array([1.0,    0.15,   25.0e-6,  0.200]),
    'param_names':  ['SO2', 'CBV', 'R', 'T2'],

    # --- GESFIDE sequence geometry ---
    'n_fid':    14,   # Part A: echoes 0-13
    'n_rephas': 16,   # Part B: echoes 14-29  (SE at echo 29)
    # Part C: echoes 30-39 (post-SE)

    # --- Dual-regime feature scaling  ---
    # R2*_A range: physiologically [5, 50] s^-1 (R2=5..20, R2'=0..30)
    # R2*_B range: [R2 - R2'] can be as negative as ~-25 s^-1
    #   we scale each independently to [0, 1]
    'R2starA_min': 2.0,    # s^-1  (hard lower bound; R2_min = 5 → R2+R2' >= 5)
    'R2starA_max': 55.0,   # s^-1  (R2_max=20 + R2'_max~35)
    'R2starB_min': -30.0,  # s^-1  (negative when R2' > R2)
    'R2starB_max': 22.0,   # s^-1  (R2_max with near-zero R2')

    # --- Training ---
    'snr_levels':    [20, 50, 100, 150],
    'n_samples':     1_600_000,
    'test_frac':     0.15,
    'val_frac':      0.15,
    'batch_size':    16384,
    'epochs':        200,
    'patience':      25,
    'lr':            5e-5,
    'dropout':       0.05,
    # Weighted loss: up-weight CBV and R which have small dynamic ranges
    'param_weights': [1.0, 12.0, 8.0, 1.0],   # [SO2, CBV, R, T2]
}

SE_ECHO = CONFIG['n_fid'] + CONFIG['n_rephas']   # = 30  (first post-SE echo; echo 29 = last of Part B)
os.makedirs(CONFIG['output_dir'], exist_ok=True)
os.makedirs(os.path.join(CONFIG['output_dir'], 'models'), exist_ok=True)

print(f'SE_ECHO index = {SE_ECHO}  (echo 29 = last of Part B = spin echo)')
print(f'Input dim: 40 + 2 = 42  (signal + R2*_A + R2*_B)')
print(f'Output dim: 4  ({CONFIG["param_names"]})')

## 3. Load echo times

In [ ]:
et_mat       = sio.loadmat(CONFIG['echotimes_path'])
echo_times_s = et_mat['Echotimes'].flatten() / 1000.0   # convert ms → s

T_A = echo_times_s[:CONFIG['n_fid']]            # Part A absolute times
T_B = echo_times_s[CONFIG['n_fid']:SE_ECHO]     # Part B absolute times

print(f'Part A: echoes  0–{CONFIG["n_fid"]-1},  t = [{T_A[0]*1e3:.2f}, ..., {T_A[-1]*1e3:.2f}] ms')
print(f'Part B: echoes {CONFIG["n_fid"]}–{SE_ECHO-1},  t = [{T_B[0]*1e3:.2f}, ..., {T_B[-1]*1e3:.2f}] ms')
print(f'  → Spin echo (echo {SE_ECHO-1}) at t = {T_B[-1]*1e3:.2f} ms  (should be ~100 ms)')

## 4. Utilities

In [ ]:
def load_mat(path, key):
    """Load .mat v5 or v7.3 (HDF5) — auto-detects format."""
    try:
        mat = sio.loadmat(path)
        if key in mat:
            return np.array(mat[key], dtype=np.float32)
        cands = [k for k in mat if not k.startswith('_')]
        print(f'  Key "{key}" not found, using "{cands[0]}"')
        return np.array(mat[cands[0]], dtype=np.float32)
    except NotImplementedError:
        with h5py.File(path, 'r') as f:
            data = f[key][()] if key in f else f[next(k for k in f if not k.startswith('#'))][()]
            if data.ndim >= 2:
                data = data.T
            return np.array(data, dtype=np.float32)


def euclidean_norm(data):
    """L2-normalise rows of signal matrix."""
    data = np.abs(data).astype(np.float32)
    norms = np.linalg.norm(data, axis=1, keepdims=True)
    return data / np.maximum(norms, 1e-12)


def params_scale(params, mins, maxs):
    return ((params - mins) / (maxs - mins)).astype(np.float32)


def params_inverse(scaled, mins, maxs):
    return (scaled * (maxs - mins) + mins).astype(np.float32)


def filter_param_range(signals, params, mins, maxs):
    mask = np.ones(len(params), dtype=bool)
    for i in range(min(params.shape[1], len(mins))):
        mask &= (params[:, i] >= mins[i]) & (params[:, i] <= maxs[i])
    n_removed = (~mask).sum()
    if n_removed:
        print(f'  Filtered {n_removed} out-of-range entries ({n_removed/len(mask)*100:.1f}%)')
    return signals[mask], params[mask]


def clean_data(signals, params):
    valid = np.all(np.isfinite(signals), axis=1) & np.all(np.isfinite(params), axis=1)
    n_removed = (~valid).sum()
    if n_removed:
        print(f'  Removed {n_removed} non-finite entries')
    return signals[valid], params[valid]


print('✓ Utilities ready')

## 5. Dual-regime feature computation

For each voxel/sample, we compute two features from the **raw (un-normalised)** signal:

**Feature 1 — R2*_A** (always positive, Part A always decays):  
Fit `log S_A(t) = log(S0) − R2*_A · t` via OLS on Part A echoes.  
R2*_A = −slope_A.  Encodes **R2 + R2'** → gives a lower bound on T2.

**Feature 2 — R2*_B** (can be negative when R2' > R2):  
Fit `log S_B(t) = log(S0_B) − R2*_B · t` via OLS on Part B echoes.  
R2*_B = −slope_B. **No clipping** — the sign carries information about whether Part B
is decaying (R2' < R2) or rephasing (R2' > R2).  Encodes **R2 − R2'**.

Together:  R2 = (R2*_A + R2*_B)/2,  R2' = (R2*_A − R2*_B)/2

In [ ]:
def ols_slope(t_vec, sig_mat):
    """
    OLS slope of log(|S|) vs absolute echo time t.
    Returns raw slope (negative for decaying, positive for rephasing).
    Shape: sig_mat (N, K), t_vec (K,) → returns (N,)
    """
    log_s  = np.log(np.maximum(np.abs(sig_mat), 1e-9)).astype(np.float64)
    t      = t_vec.astype(np.float64)
    t_c    = t - t.mean()
    log_sm = log_s - log_s.mean(axis=1, keepdims=True)
    return (log_sm * t_c[None, :]).sum(axis=1) / (t_c ** 2).sum()  # (N,)


def compute_dual_regime_features(sig_raw, config):
    """
    Compute R2*_A and R2*_B from the raw (un-normalised) signal.

    Physical meaning
    ----------------
    R2*_A = R2 + R2'   (Part A: FID decay, always positive)
    R2*_B = R2 − R2'   (Part B: rephasing, can be negative when R2' > R2)

    Returns
    -------
    R2starA : (N,) float32, s^-1, always >= 0 (clipped at lower bound)
    R2starB : (N,) float32, s^-1, raw signed value — NOT clipped
    """
    n_fid   = config['n_fid']
    se_echo = n_fid + config['n_rephas']   # = 30

    # Part A OLS — negate slope: signal decays → slope < 0 → R2*_A = −slope > 0
    slope_A = ols_slope(T_A, sig_raw[:, :n_fid])
    R2starA = np.maximum(-slope_A, 1.0 / config['param_maxs'][3])  # clip lower bound only

    # Part B OLS — negate slope: can be positive (refocusing) or negative (decaying)
    # IMPORTANT: do NOT clip — sign encodes whether R2' < or > R2
    slope_B = ols_slope(T_B, sig_raw[:, n_fid:se_echo])
    R2starB = -slope_B   # raw signed value

    return R2starA.astype(np.float32), R2starB.astype(np.float32)


def scale_dual_features(R2starA, R2starB, config):
    """
    Min-max scale both regime features to [0, 1] using configured ranges.
    Values outside the range are clipped.
    """
    a_min, a_max = config['R2starA_min'], config['R2starA_max']
    b_min, b_max = config['R2starB_min'], config['R2starB_max']

    feat_A = np.clip((R2starA - a_min) / (a_max - a_min), 0.0, 1.0).astype(np.float32)
    feat_B = np.clip((R2starB - b_min) / (b_max - b_min), 0.0, 1.0).astype(np.float32)
    return feat_A, feat_B


print('✓ Dual-regime feature functions defined')

## 6. Diagnostic: validate features on noise-free signals

Check that R2*_A and R2*_B bracket the true T2, and that their combination
recovers R2 and R2' correctly on noise-free data.

In [ ]:
print('Loading noise-free signals for diagnostic...')
nf_sig    = load_mat(CONFIG['noisefree_sig_path'], CONFIG['dict_key'])
nf_params = load_mat(CONFIG['param_path'],         CONFIG['param_key'])[:, :4]
nf_sig, nf_params = filter_param_range(
    nf_sig, nf_params, CONFIG['param_mins'], CONFIG['param_maxs'])
nf_sig, nf_params = clean_data(nf_sig, nf_params)

rng = np.random.default_rng(42)
idx = rng.choice(len(nf_sig), 50_000, replace=False)
sig_v  = nf_sig[idx]
par_v  = nf_params[idx]

# Compute dual features on noise-free signals
R2starA_nf, R2starB_nf = compute_dual_regime_features(sig_v, CONFIG)

# True values from simulation
T2_true  = par_v[:, 3] * 1000.0    # ms
R2_true  = 1.0 / par_v[:, 3]       # s^-1

# Derived R2 and R2' from the two features
R2_combined   = (R2starA_nf + R2starB_nf) / 2.0
R2prime_est   = (R2starA_nf - R2starB_nf) / 2.0
T2_combined   = np.clip(1000.0 / np.maximum(R2_combined, 1.0), 50, 200)  # ms
T2_A_proxy    = np.clip(1000.0 / R2starA_nf, 0, 500)   # always < true T2

# Summary statistics
def rmse(a, b): return float(np.sqrt(np.mean((a - b)**2)))
def bias(a, b): return float(np.mean(a - b))

print(f'\n--- Noise-free diagnostic (N={len(sig_v):,}) ---')
print(f'R2*_A  (= R2 + R2\'):  range [{R2starA_nf.min():.1f}, {R2starA_nf.max():.1f}] s^-1')
print(f'R2*_B  (= R2 - R2\'):  range [{R2starB_nf.min():.1f}, {R2starB_nf.max():.1f}] s^-1  (negative = Part B rephasing)')
print(f'Fraction with R2*_B < 0 (Part B rephasing): {(R2starB_nf < 0).mean()*100:.1f}%')
print()
print(f'T2_A proxy (1/R2*_A):     RMSE={rmse(T2_A_proxy, T2_true):.2f}ms  Bias={bias(T2_A_proxy, T2_true):+.2f}ms  [expect negative bias]')
print(f'T2 from combined (Ni):    RMSE={rmse(T2_combined, T2_true):.2f}ms  Bias={bias(T2_combined, T2_true):+.2f}ms  [should be low on noise-free]')
print()
print(f'R2_combined (noise-free): RMSE={rmse(R2_combined, R2_true):.4f} s^-1  vs. true R2')
print(f'R2prime_est (noise-free): range [{R2prime_est.min():.1f}, {R2prime_est.max():.1f}] s^-1')

# Plots
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].scatter(T2_true[:3000], T2_A_proxy[:3000], s=1, alpha=0.3)
axes[0].plot([50,200],[50,200],'r--')
axes[0].set_xlabel('True T2 (ms)'); axes[0].set_ylabel('T2_A proxy (ms)')
axes[0].set_title('T2_A = 1/R2*_A  (lower bound)')

axes[1].scatter(T2_true[:3000], T2_combined[:3000], s=1, alpha=0.3, c='orange')
axes[1].plot([50,200],[50,200],'r--')
axes[1].set_xlabel('True T2 (ms)'); axes[1].set_ylabel('T2 combined (ms)')
axes[1].set_title('T2 from (R2*_A + R2*_B)/2  (noise-free)')

axes[2].hist(R2starB_nf, bins=80, color='steelblue', edgecolor='none')
axes[2].axvline(0, color='r', lw=1.5, label='R2*_B = 0')
axes[2].set_xlabel('R2*_B (s⁻¹)'); axes[2].set_ylabel('Count')
axes[2].set_title('R2*_B distribution  (negative = Part B rephasing)')
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'diagnostic_dual_features_noisefree.png'), dpi=150)
plt.show()
print('Diagnostic plot saved.')

## 7. Diagnostic: validate features on noisy signals

Repeat the diagnostic at the clinical SNR level (SNR=50) to confirm features
are still informative despite noise.

In [ ]:
snr_diag = 50
noisy_sig_path = os.path.join(CONFIG['dict_base_path'], f'QuasiRand_t2_snr{snr_diag}.mat')
print(f'Loading SNR={snr_diag} signals...')
sig_noisy = load_mat(noisy_sig_path, CONFIG['dict_key'])
sig_noisy, params_noisy = filter_param_range(
    sig_noisy, load_mat(CONFIG['param_path'], CONFIG['param_key'])[:len(sig_noisy), :4],
    CONFIG['param_mins'], CONFIG['param_maxs'])
sig_noisy, params_noisy = clean_data(sig_noisy, params_noisy)

idx_n = rng.choice(len(sig_noisy), 50_000, replace=False)
sig_n  = sig_noisy[idx_n]
par_n  = params_noisy[idx_n]

R2starA_n, R2starB_n = compute_dual_regime_features(sig_n, CONFIG)

T2_true_n   = par_n[:, 3] * 1000.0
R2_true_n   = 1.0 / par_n[:, 3]
R2_comb_n   = (R2starA_n + R2starB_n) / 2.0
T2_comb_n   = np.clip(1000.0 / np.maximum(R2_comb_n, 1.0), 50, 200)
T2_A_n      = np.clip(1000.0 / R2starA_n, 0, 500)

feat_A_n, feat_B_n = scale_dual_features(R2starA_n, R2starB_n, CONFIG)

print(f'\n--- SNR={snr_diag} diagnostic (N={len(sig_n):,}) ---')
print(f'R2*_A: range [{R2starA_n.min():.1f}, {R2starA_n.max():.1f}] s^-1')
print(f'R2*_B: range [{R2starB_n.min():.1f}, {R2starB_n.max():.1f}] s^-1')
print(f'Fraction with R2*_B < 0: {(R2starB_n < 0).mean()*100:.1f}%')
print(f'T2_A proxy:  RMSE={rmse(T2_A_n,   T2_true_n):.2f}ms  Bias={bias(T2_A_n,   T2_true_n):+.2f}ms')
print(f'T2 combined: RMSE={rmse(T2_comb_n, T2_true_n):.2f}ms  Bias={bias(T2_comb_n, T2_true_n):+.2f}ms')
print(f'Scaled feat_A in [0,1]: min={feat_A_n.min():.3f}  max={feat_A_n.max():.3f}  mean={feat_A_n.mean():.3f}')
print(f'Scaled feat_B in [0,1]: min={feat_B_n.min():.3f}  max={feat_B_n.max():.3f}  mean={feat_B_n.mean():.3f}')

## 8. Dataset preparation

For each SNR level, compute dual-regime features from the noisy signal **before** L2 normalisation
(amplitude information is needed for R2*_B computation). Then L2-normalise and concatenate.

**Input layout**: `[L2-norm signal (40)] + [feat_A (1)] + [feat_B (1)]`  →  shape `(N, 42)`

In [ ]:
def prepare_dual_regime_dataset(config):
    print('=' * 65)
    print('BUILDING DUAL-REGIME MIXED-SNR DATASET (42-dim input)')
    print('=' * 65)

    params_raw = load_mat(config['param_path'], config['param_key'])[:, :4]
    all_x, all_y = [], []

    for snr in config['snr_levels']:
        print(f'\n  SNR={snr}...')
        sig_path = os.path.join(config['dict_base_path'], f'QuasiRand_t2_snr{snr}.mat')
        sig_raw  = load_mat(sig_path, config['dict_key'])
        sig_i, par_i = filter_param_range(
            sig_raw, params_raw.copy(),
            config['param_mins'], config['param_maxs'])

        # ── Step 1: compute dual-regime features from RAW (un-normalised) signal ──
        R2starA, R2starB = compute_dual_regime_features(sig_i, config)
        feat_A, feat_B   = scale_dual_features(R2starA, R2starB, config)

        # ── Step 2: L2-normalise signal (removes absolute amplitude) ──
        sig_norm, par_i = clean_data(euclidean_norm(sig_i), par_i)

        # Ensure consistent length after clean_data
        n = len(par_i)
        feat_A = feat_A[:n]
        feat_B = feat_B[:n]

        # ── Step 3: assemble 42-dim input ──
        x_i = np.concatenate([
            sig_norm,           # (N, 40)
            feat_A[:, None],    # (N,  1)  R2*_A scaled
            feat_B[:, None],    # (N,  1)  R2*_B scaled
        ], axis=1)              # (N, 42)

        # ── Step 4: scale outputs to [0,1] ──
        y_i = params_scale(par_i[:, :4],
                           config['param_mins'][:4],
                           config['param_maxs'][:4])

        all_x.append(x_i)
        all_y.append(y_i)
        print(f'    Loaded {n:,} samples | '
              f'feat_A [{feat_A.min():.2f},{feat_A.max():.2f}] | '
              f'feat_B [{feat_B.min():.2f},{feat_B.max():.2f}]')

    X = np.vstack(all_x)
    Y = np.vstack(all_y)

    # Down-sample to requested size
    if len(X) > config['n_samples']:
        idx = np.random.choice(len(X), config['n_samples'], replace=False)
        X, Y = X[idx], Y[idx]

    print(f'\nTotal: {len(X):,} samples | input_dim={X.shape[1]} | output_dim={Y.shape[1]}')

    x_tv, x_test, y_tv, y_test = train_test_split(
        X, Y, test_size=config['test_frac'], random_state=42)
    x_train, x_val, y_train, y_val = train_test_split(
        x_tv, y_tv, test_size=config['val_frac'], random_state=42)

    y_test_raw = params_inverse(y_test,
                                config['param_mins'][:4],
                                config['param_maxs'][:4])

    print(f'  Train={len(x_train):,} | Val={len(x_val):,} | Test={len(x_test):,}')
    return x_train, x_val, x_test, y_train, y_val, y_test, y_test_raw


(x_train, x_val, x_test,
 y_train, y_val,
 y_test,  y_test_raw) = prepare_dual_regime_dataset(CONFIG)

print(f'\nInput dim:  {x_train.shape[1]}  (expected 42)')
print(f'Output dim: {y_train.shape[1]}  (expected 4)')
print(f'feat_A feature range: [{x_train[:,40].min():.3f}, {x_train[:,40].max():.3f}]')
print(f'feat_B feature range: [{x_train[:,41].min():.3f}, {x_train[:,41].max():.3f}]')

## 9. Model architecture

Conv1D backbone with **two-scalar FiLM conditioning** (one per regime feature).

FiLM (Feature-wise Linear Modulation) applies per-channel scaling and shifting based
on the conditioning scalars, letting the model adaptively re-weight convolutional
features based on regime-derived physics. Each FiLM layer receives **both** feat_A
and feat_B as a 2-dimensional conditioning vector.

```
Input:  (N, 42) = 40-echo signal + feat_A + feat_B
        ↓
Conv1D backbone on echoes [:40] → 1280-dim feature
        ↓
FC layers with FiLM conditioning from (feat_A, feat_B) concatenated
        ↓
Output: (N, 4) = SO2, CBV, R, T2 in [0,1]
```

In [ ]:
class Clamp01(nn.Module):
    def forward(self, x): return x.clamp(0.0, 1.0)


class FiLMLayer(nn.Module):
    """
    Feature-wise Linear Modulation: y = gamma(cond) * x + beta(cond)
    Accepts a conditioning vector of any size (here 2-dim for feat_A + feat_B).
    """
    def __init__(self, feature_dim, cond_in=2, cond_hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_in, cond_hidden), nn.ReLU(),
            nn.Linear(cond_hidden, 2 * feature_dim),
        )
        # Init: gamma=1 (identity), beta=0 → no modulation at start
        nn.init.zeros_(self.net[-1].weight)
        bias_init = torch.zeros(2 * feature_dim)
        bias_init[:feature_dim] = 1.0
        self.net[-1].bias.data.copy_(bias_init)
        self.feature_dim = feature_dim

    def forward(self, x, cond_vec):
        # cond_vec: (B, cond_in)
        params = self.net(cond_vec)           # (B, 2*feature_dim)
        gamma  = params[:, :self.feature_dim]   # (B, feature_dim)
        beta   = params[:, self.feature_dim:]   # (B, feature_dim)
        return gamma * x + beta


class DualRegimeModel(nn.Module):
    """
    Conv1D backbone + FiLM conditioning on dual-regime features (R2*_A, R2*_B).

    Input:  (B, 42) = 40-echo L2-norm signal + feat_A + feat_B
    Output: (B,  4) = SO2, CBV, R, T2 in [0,1]
    """
    def __init__(self, n_outputs=4, dropout=0.05, film_cond_hidden=32):
        super().__init__()

        # Conv1D extracts temporal features from the 40-echo decay curve
        # Spatial pooling: 40 → 20 → 10 → 5   conv_out = 256 * 5 = 1280
        self.conv = nn.Sequential(
            nn.Conv1d(1, 32,  kernel_size=7, padding=3),
            nn.BatchNorm1d(32),  nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(32, 64,  kernel_size=5, padding=2),
            nn.BatchNorm1d(64),  nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256), nn.ReLU(),
        )

        # Three FC layers, each modulated by FiLM using the 2-dim conditioning vector
        self.fc1   = nn.Linear(1280, 512);  self.bn1 = nn.BatchNorm1d(512)
        self.film1 = FiLMLayer(512,  cond_in=2, cond_hidden=film_cond_hidden)
        self.fc2   = nn.Linear(512,  256);  self.bn2 = nn.BatchNorm1d(256)
        self.film2 = FiLMLayer(256,  cond_in=2, cond_hidden=film_cond_hidden)
        self.fc3   = nn.Linear(256,  128);  self.bn3 = nn.BatchNorm1d(128)
        self.film3 = FiLMLayer(128,  cond_in=2, cond_hidden=film_cond_hidden)

        self.fc_out  = nn.Linear(128, n_outputs)
        self.out_act = Clamp01()
        self.drop    = nn.Dropout(dropout)
        self.relu    = nn.ReLU()

    def forward(self, x):
        # Split input into signal and regime features
        echo = x[:, :40].unsqueeze(1)    # (B, 1, 40) for Conv1D
        cond = x[:, 40:42]               # (B, 2)  [feat_A, feat_B]

        # Conv backbone
        c = self.conv(echo).flatten(1)   # (B, 1280)

        # FC layers with FiLM conditioning
        h = self.drop(self.relu(self.bn1(self.film1(self.fc1(c), cond))))
        h = self.drop(self.relu(self.bn2(self.film2(self.fc2(h), cond))))
        h = self.drop(self.relu(self.bn3(self.film3(self.fc3(h), cond))))
        return self.out_act(self.fc_out(h))


# Sanity check
_m = DualRegimeModel(n_outputs=4).to(device)
_x = torch.randn(8, 42).to(device)
print(f'Output shape: {_m(_x).shape}   (expected [8, 4])')
print(f'Model parameters: {sum(p.numel() for p in _m.parameters()):,}')
del _m, _x

## 10. Loss function and training loop

In [ ]:
class WeightedMAELoss(nn.Module):
    """Weighted MAE loss to balance parameters with different dynamic ranges."""
    def __init__(self, param_weights):
        super().__init__()
        self.register_buffer('w', torch.tensor(param_weights, dtype=torch.float32))

    def forward(self, pred, target):
        per_param  = torch.abs(pred - target) * self.w.unsqueeze(0)  # (B, 4)
        return per_param.sum(dim=1).mean()


def train_model(model, x_train, y_train, x_val, y_val, config):
    criterion = WeightedMAELoss(config['param_weights']).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=8, factor=0.5, min_lr=1e-7, verbose=True)

    # DataLoaders
    x_t = torch.tensor(x_train, dtype=torch.float32).to(device)
    y_t = torch.tensor(y_train, dtype=torch.float32).to(device)
    x_v = torch.tensor(x_val,   dtype=torch.float32).to(device)
    y_v = torch.tensor(y_val,   dtype=torch.float32).to(device)

    train_loader = DataLoader(
        TensorDataset(x_t, y_t), batch_size=config['batch_size'], shuffle=True)

    best_val   = float('inf')
    patience_cnt = 0
    history    = {'train_loss': [], 'val_loss': []}
    model_path = os.path.join(config['output_dir'], 'models', 'dual_regime_best.pt')

    print(f'Training for up to {config["epochs"]} epochs  (patience={config["patience"]})')
    print(f'Train batches per epoch: {len(train_loader):,}')

    for epoch in range(1, config['epochs'] + 1):
        # -- Train --
        model.train()
        t0  = time.time()
        tloss = 0.0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            tloss += loss.item() * len(xb)
        tloss /= len(x_t)

        # -- Validate --
        model.eval()
        with torch.no_grad():
            vloss = criterion(model(x_v), y_v).item()

        scheduler.step(vloss)
        history['train_loss'].append(tloss)
        history['val_loss'].append(vloss)

        # -- Early stopping --
        if vloss < best_val:
            best_val = vloss
            patience_cnt = 0
            torch.save(model.state_dict(), model_path)
        else:
            patience_cnt += 1

        if epoch % 10 == 0 or epoch <= 5:
            lr = optimizer.param_groups[0]['lr']
            print(f'  Epoch {epoch:4d}/{config["epochs"]}  '
                  f'train={tloss:.5f}  val={vloss:.5f}  '
                  f'best={best_val:.5f}  lr={lr:.2e}  '
                  f'({time.time()-t0:.1f}s)')

        if patience_cnt >= config['patience']:
            print(f'Early stop at epoch {epoch}  (patience={config["patience"]})')
            break

    # Load best
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f'\nBest val loss: {best_val:.6f}  — model loaded from {model_path}')
    return model, history


print('✓ Loss and training loop ready')

## 11. Train the model

In [ ]:
model = DualRegimeModel(
    n_outputs=4,
    dropout=CONFIG['dropout'],
    film_cond_hidden=32
).to(device)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

model, history = train_model(
    model, x_train, y_train, x_val, y_val, CONFIG)

## 12. Training curve

In [ ]:
plt.figure(figsize=(8, 4))
epochs_done = len(history['train_loss'])
plt.plot(range(1, epochs_done+1), history['train_loss'], label='Train')
plt.plot(range(1, epochs_done+1), history['val_loss'],   label='Val')
plt.xlabel('Epoch'); plt.ylabel('Weighted MAE loss')
plt.title('Dual-Regime T2 Model — Training Curve')
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'training_curve.png'), dpi=150)
plt.show()

# Save training history
with open(os.path.join(CONFIG['output_dir'], 'training_history.json'), 'w') as f:
    json.dump({'train_loss': history['train_loss'],
               'val_loss':   history['val_loss']}, f, indent=2)

## 13. Evaluation on held-out test set

In [ ]:
# Restart CUDA cleanly
torch.cuda.empty_cache()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

In [ ]:
def batched_predict(model, x_np, batch_size=4096, dev=device):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(x_np), batch_size):
            xb = torch.tensor(
                x_np[i:i+batch_size], dtype=torch.float32
            ).to(dev).contiguous()
            preds.append(model(xb).cpu().numpy())
    return np.concatenate(preds, axis=0)

In [ ]:
torch.cuda.empty_cache()
model.eval()
batch_size = 8192
preds = []
with torch.no_grad():
    for i in range(0, len(x_test), batch_size):
        xb = torch.tensor(x_test[i:i+batch_size], dtype=torch.float32).to(device).contiguous()
        preds.append(model(xb).cpu().numpy())
y_pred_scaled = np.concatenate(preds, axis=0)

y_pred_raw = params_inverse(y_pred_scaled,
                            CONFIG['param_mins'][:4],
                            CONFIG['param_maxs'][:4])

UNITS  = ['%',  '%',  'µm',    'ms']
SCALES = [100., 100., 1e6,     1000.]
PNAMES = CONFIG['param_names']

results = {}
print(f"\n{'='*70}")
print(f"{'Parameter':<10} {'RMSE':>10} {'Bias':>10} {'Pearson r':>12} {'R²':>8}")
print(f"{'-'*70}")
for i, (name, unit, sc) in enumerate(zip(PNAMES, UNITS, SCALES)):
    pred = y_pred_raw[:, i] * sc
    true = y_test_raw[:, i] * sc
    diff = pred - true
    r    = float(np.corrcoef(true, pred)[0, 1])
    r2   = float(r2_score(true, pred))
    rms  = float(np.sqrt(np.mean(diff**2)))
    bias_v = float(np.mean(diff))
    results[name] = {'rmse': rms, 'bias': bias_v, 'r': r, 'r2': r2, 'unit': unit}
    print(f"{name:<10} {rms:>8.3f}{unit:>2} {bias_v:>+9.3f}{unit:>2} {r:>12.4f} {r2:>8.4f}")
print(f"{'='*70}")

with open(os.path.join(CONFIG['output_dir'], 'test_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

## 14. Scatter plots: predicted vs. true (test set)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Dual-Regime T2 Model — Test Set Predictions vs. Ground Truth',
             fontsize=11, fontweight='bold')

subsample = min(10000, len(y_pred_raw))
idx_plot  = np.random.choice(len(y_pred_raw), subsample, replace=False)

for i, (name, unit, sc) in enumerate(zip(PNAMES, UNITS, SCALES)):
    ax   = axes[i]
    pred = y_pred_raw[idx_plot, i] * sc
    true = y_test_raw[idx_plot, i] * sc
    lo   = min(true.min(), pred.min())
    hi   = max(true.max(), pred.max())

    ax.scatter(true, pred, s=1, alpha=0.2, rasterized=True)
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='identity')
    r2v = results[name]['r2']
    ax.set_title(f'{name}  (R²={r2v:.3f})')
    ax.set_xlabel(f'True {name} ({unit})')
    ax.set_ylabel(f'Predicted {name} ({unit})')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'scatter_test.png'), dpi=150)
plt.show()

## 15. RMSE boxplots per SNR level

Evaluate each SNR level independently to see where the model excels and where it struggles.

In [ ]:
per_snr_rmse = {name: {} for name in PNAMES}

params_raw_all = load_mat(CONFIG['param_path'], CONFIG['param_key'])[:, :4]

for snr in CONFIG['snr_levels']:
    print(f'  Evaluating SNR={snr}...', end=' ')
    sig_path = os.path.join(CONFIG['dict_base_path'], f'QuasiRand_t2_snr{snr}.mat')
    sig_raw  = load_mat(sig_path, CONFIG['dict_key'])
    sig_i, par_i = filter_param_range(
        sig_raw, params_raw_all.copy(),
        CONFIG['param_mins'], CONFIG['param_maxs'])

    # Build 42-dim input for this SNR (same pipeline as training)
    R2starA_i, R2starB_i = compute_dual_regime_features(sig_i, CONFIG)
    feat_A_i, feat_B_i   = scale_dual_features(R2starA_i, R2starB_i, CONFIG)
    sig_norm_i, par_i    = clean_data(euclidean_norm(sig_i), par_i)
    n = len(par_i)
    x_eval = np.concatenate([sig_norm_i,
                              feat_A_i[:n, None],
                              feat_B_i[:n, None]], axis=1).astype(np.float32)

    # Predict
    model.eval()
    with torch.no_grad():
        # preds = model(torch.tensor(x_eval).to(device)).cpu().numpy()
        preds = batched_predict(model, x_eval)
    preds_raw = params_inverse(preds, CONFIG['param_mins'][:4], CONFIG['param_maxs'][:4])

    # RMSE per parameter
    for j, (name, sc) in enumerate(zip(PNAMES, SCALES)):
        per_snr_rmse[name][snr] = float(
            np.sqrt(np.mean((preds_raw[:, j]*sc - par_i[:, j]*sc)**2)))
    print(f'done  (N={n:,})')

# Print table
print(f'\n{"":>10}', end='')
for snr in CONFIG['snr_levels']: print(f'  SNR={snr:>3}', end='')
print()
for name, unit in zip(PNAMES, UNITS):
    print(f'{name+" ("+unit+")":<14}', end='')
    for snr in CONFIG['snr_levels']:
        print(f'  {per_snr_rmse[name][snr]:>8.3f}', end='')
    print()

# Bar chart
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
snr_labels = [str(s) for s in CONFIG['snr_levels']]
for i, (name, unit) in enumerate(zip(PNAMES, UNITS)):
    vals = [per_snr_rmse[name][s] for s in CONFIG['snr_levels']]
    axes[i].bar(snr_labels, vals, color='steelblue', edgecolor='white')
    axes[i].set_xlabel('SNR'); axes[i].set_ylabel(f'RMSE ({unit})')
    axes[i].set_title(name)

fig.suptitle('Dual-Regime Model — RMSE per SNR', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'rmse_per_snr.png'), dpi=150)
plt.show()

## 16. Feature importance: are both regime features useful?

Ablation: evaluate the model with each regime feature set to 0.5 (neutral/uninformative)
to measure the isolated contribution of each feature.

In [ ]:
x_ts_np = x_test.copy()
y_ts_np = y_test_raw

conditions = {
    'Full (feat_A + feat_B)': x_ts_np.copy(),
    'Ablate feat_A (→0.5)':   x_ts_np.copy(),
    'Ablate feat_B (→0.5)':   x_ts_np.copy(),
    'Ablate both (→0.5)':     x_ts_np.copy(),
}
conditions['Ablate feat_A (→0.5)'][:,  40] = 0.5
conditions['Ablate feat_B (→0.5)'][:,  41] = 0.5
conditions['Ablate both (→0.5)'][:,  40] = 0.5
conditions['Ablate both (→0.5)'][:,  41] = 0.5

model.eval()
print('\nAblation study — RMSE degradation when regime features are zeroed out:\n')
print(f'{"Condition":<30}', end='')
for name, unit in zip(PNAMES, UNITS): print(f'  {name+"("+unit+")":>12}', end='')
print()
print('-' * 80)

for cond_name, x_cond in conditions.items():
    # with torch.no_grad():
    #     preds = model(torch.tensor(x_cond, dtype=torch.float32).to(device)).cpu().numpy()
    preds = batched_predict(model, x_cond)
    preds_raw = params_inverse(preds, CONFIG['param_mins'][:4], CONFIG['param_maxs'][:4])
    print(f'{cond_name:<30}', end='')
    for j, (name, sc) in enumerate(zip(PNAMES, SCALES)):
        rms = float(np.sqrt(np.mean((preds_raw[:, j]*sc - y_ts_np[:, j]*sc)**2)))
        print(f'  {rms:>12.3f}', end='')
    print()

## 17. Save model and config

In [ ]:
# Save final model
final_path = os.path.join(CONFIG['output_dir'], 'models', 'dual_regime_final.pt')
torch.save({
    'model_state_dict': model.state_dict(),
    'config':           {k: v.tolist() if isinstance(v, np.ndarray) else v
                         for k, v in CONFIG.items()},
    'results':          results,
    'per_snr_rmse':     per_snr_rmse,
    'input_dim':        42,
    'input_layout':     '40 L2-norm echoes + feat_A (R2*_A scaled) + feat_B (R2*_B scaled)',
    'feature_scaling':  {
        'R2starA': {'min': CONFIG['R2starA_min'], 'max': CONFIG['R2starA_max']},
        'R2starB': {'min': CONFIG['R2starB_min'], 'max': CONFIG['R2starB_max']},
    },
}, final_path)

print(f'Model saved: {final_path}')
print()
print('To load and run inference:')
print('  ckpt = torch.load("dual_regime_final.pt")')
print('  model = DualRegimeModel(n_outputs=4)')
print('  model.load_state_dict(ckpt["model_state_dict"])')
print('  # Build 42-dim input:')
print('  R2starA, R2starB = compute_dual_regime_features(sig_raw, CONFIG)')
print('  feat_A, feat_B   = scale_dual_features(R2starA, R2starB, CONFIG)')
print('  x = np.concatenate([euclidean_norm(sig_raw), feat_A[:,None], feat_B[:,None]], axis=1)')

---
## Summary

### Why this approach is physically grounded

| Feature | Physics | Range | Information carried |
|---------|---------|-------|--------------------|
| feat_A (R2*_A scaled) | From Part A OLS, `= R2 + R2'` | always > 0 | **Lower bound** on T2; encodes combined relaxation |
| feat_B (R2*_B scaled) | From Part B OLS, `= R2 − R2'` | can be negative | **Sign indicates** whether Part B decays or rephases; encodes reversible dephasing |

Together they give the network independent access to both R2 and R2',  
eliminating the systematic underestimation that arises from combining them into
a single biased T2 estimate (Ni Eq. 3 fails in practice when R2' > R2).

The **spread** `feat_A − feat_B ∝ R2'` directly encodes the oxygen-sensitive  
susceptibility component, giving the network an explicit physics-derived cue  
for SO₂ and CBV estimation.

### Comparison to prior approaches

| Approach | Additional inputs | T2 estimation quality |
|----------|------------------|----------------------|
| Option 3 (R2_eff) | 1 scalar: `(logS0 − logS_SE)/T_SE` | Encodes R2 + residual R2'_irrev; no sign information |
| Ni Eq. 3 T2 | 1 scalar: `1/((R2*_A + R2*_B)/2)` | Systematically biased when Part B rephases |
| **This notebook** | 2 scalars: R2*_A, R2*_B separately | R2' and R2 independently accessible; richer physics |
